<a href="https://colab.research.google.com/github/jconomanp-hub/agenteIA_Consultas/blob/main/ProyALura_JuanCo%C3%B1oman_Agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalamos las librerías necesarias para el procesamiento de PDF, CSV, embeddings, base de datos vectorial y LangChain!

In [8]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf pandas streamlit



##Configuración de la API Key de Gemini

In [2]:
import os
from google.colab import userdata

# Recuperamos la clave de manera segura de los secretos de Colab
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ API Key de Gemini cargada correctamente.")
except Exception as e:
    print("❌ No se encontró la API Key en los secretos de Colab. Asegúrate de configurar 'GEMINI_API_KEY'.")


✅ API Key de Gemini cargada correctamente.


##Procesamiento del Documento (Lectura y Fragmentación)

In [3]:
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configura la ruta de tu archivo de prueba aquí:
FILE_PATH = "futureapplication.pdf"  # O "datos.csv"

def cargar_documento(file_path):
    ext = file_path.split('.')[-1].lower()
    if ext == 'pdf':
        loader = PyPDFLoader(file_path)
        documents = loader.load()
        print(f"📖 PDF cargado. Páginas leídas: {len(documents)}")
    elif ext == 'csv':
        loader = CSVLoader(file_path)
        documents = loader.load()
        print(f"📊 CSV cargado. Registros leídos: {len(documents)}")
    else:
        raise ValueError("Formato de archivo no soportado. Usa PDF o CSV.")
    return documents

# Ejecutamos la carga
documentos_brutos = cargar_documento(FILE_PATH)

# 2. Segmentamos el texto en fragmentos (Chunks) óptimos para el LLM
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documentos_brutos)
print(f"✂️ Documento fragmentado en {len(chunks)} bloques de texto.")

/tmp/ipykernel_3124/610789261.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader


📖 PDF cargado. Páginas leídas: 25
✂️ Documento fragmentado en 80 bloques de texto.


Crear la Base de Datos Vectorial (Embeddings)

In [4]:
import time
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Procesar en lotes de 50 textos para no superar el límite de 100/min
batch_size = 20
vector_store = None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"Procesando lote {i // batch_size + 1} de {(len(chunks) - 1) // batch_size + 1}...")

    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)

    # Pausa de 15 segundos entre lotes para no sobrepasar el Rate Limit
    if i + batch_size < len(chunks):
        time.sleep(15)

print("📦 Base de datos vectorial (FAISS) creada exitosamente.")

Procesando lote 1 de 4...
Procesando lote 2 de 4...
Procesando lote 3 de 4...
Procesando lote 4 de 4...
📦 Base de datos vectorial (FAISS) creada exitosamente.


Construcción de la Cadena de Consulta (RAG) y Pruebas del Agente

In [5]:
!pip install -q langchain-classic

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [7]:
# 1. Configurar el LLM (Gemini 1.5 Flash para respuestas ultrarrápidas)

# Cambia "gemini-1.5-flash" por "gemini-2.0-flash"
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)



# 2. Configurar el recuperador (Retriever)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 3. Diseñar el Prompt del Sistema
system_prompt = (
    "Eres un asistente inteligente experto en análisis de documentos internos corporativos.\n"
    "Usa los siguientes fragmentos de contexto recuperados para responder la pregunta.\n"
    "Realiza la traducción en español para entregar la respuesta.\n"
    "Si no sabes la respuesta o no está en el documento, di claramente que no dispones de esa información.\n"
    "Mantén las respuestas claras, concisas y profesionales.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Crear la cadena RAG completa
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. Prueba tu Agente con una pregunta relevante
pregunta = "¿Que es QKD?"
respuesta = rag_chain.invoke({"input": pregunta})

print(f"\n🙋‍♂️ Pregunta: {pregunta}")
print(f"🤖 Respuesta del Agente:\n{respuesta['answer']}")


🙋‍♂️ Pregunta: ¿Que es QKD?
🤖 Respuesta del Agente:
QKD (Quantum Key Distribution) es un método para intercambiar claves de cifrado que se basa en las propiedades de la mecánica cuántica, específicamente el entrelazamiento de partículas (típicamente fotones). Este proceso se realiza a través de medios ópticos (fibra terrestre, óptica de espacio libre o enlace satelital) utilizando Qubits o Unidades Cuánticas de Información.

La principal característica de QKD es que garantiza la detección inmediata si un intruso intercepta la clave utilizada para cifrar los datos. Cualquier intento de interferir con la transmisión genera una perturbación que es detectada por el protocolo de comunicación, lo que detiene la comunicación al instante y permite el envío de una nueva clave antes de que se transmita cualquier dato sensible, haciéndola esencialmente a prueba de manipulaciones.

Debido a que es una solución basada en hardware y de alto costo, QKD se utiliza principalmente en aplicaciones de al